# Microsoft Agent Framework — Azure OpenAI (Responses API)

Neste exemplo de código, você usará o **Microsoft Agent Framework (MAF)** para criar um agente simples suportado pelo **Azure OpenAI** usando a **Responses API**.

> **Nota de migração:** Este exemplo usava anteriormente o Semantic Kernel com GitHub Models. Ele foi migrado para o Microsoft Agent Framework, e o GitHub Models (obsoleto, com aposentadoria prevista para julho de 2026) foi substituído pelo Azure OpenAI, que suporta a Responses API. O `OpenAIChatClient` no MAF aponta para o endpoint estável `/openai/v1/` do Azure OpenAI e usa a Responses API por padrão.

O objetivo deste exemplo é demonstrar os passos que serão aplicados posteriormente em outros exemplos de código ao implementar vários padrões agentes.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importe os Pacotes Python Necessários


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definindo uma Ferramenta

No Microsoft Agent Framework, uma **ferramenta** é uma função Python simples decorada com `@tool` que o agente pode chamar. Abaixo definimos uma ferramenta que retorna um destino de férias aleatório e evita repetir o anterior.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Criando o Agente

Aqui, criamos o Agente chamado `TravelAgent`.

Neste exemplo, usamos instruções muito básicas. Sinta-se à vontade para modificar essas instruções para observar como o comportamento do agente muda.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Executando o Agente

Agora podemos executar o agente. Criamos uma `AgentSession` para que o agente lembre da conversa ao longo das interações, depois enviamos dois `user_inputs`. O primeiro pede uma viagem; o segundo diz que o usuário não gostou da sugestão e pede outra — o agente usa o histórico da sessão mais a ferramenta `get_random_destination` para responder.

Você pode modificar essas mensagens para observar como o agente reage de forma diferente. As respostas são **transmitidas** token por token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
